# Notebook 04 — Fine-Tuning `nlpie/tiny-biobert` for In-Hospital Mortality Prediction

This notebook fine-tunes **tiny-biobert** (`nlpie/tiny-biobert`), a compact biomedical BERT variant, on MIMIC-IV clinical discharge notes to predict **in-hospital mortality** (`hospital_expire_flag`).

The pipeline covers:
1. Environment setup and library imports
2. Data loading & preprocessing
3. Tokenisation with the tiny-biobert tokeniser
4. Class-imbalance handling via a weighted loss function
5. Model fine-tuning with the Hugging Face `Trainer`
6. Model persistence
7. NLP risk-score generation for the full cohort

## 1. Environment Setup

Enable the autoreload extension so that changes to helper modules in `../core/` are picked up automatically without restarting the kernel.

In [27]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Imports

Standard scientific-computing libraries (`pandas`, `numpy`, `torch`, `sklearn`) are imported alongside Hugging Face `transformers` and `datasets`.
Project-level helpers (`config`, `model_func`) are sourced from `../core/`.

In [28]:
import os
import sys
sys.path.append('../core')
from config import processed_data_path, results_path, raw_data_path
from model_func import save_model, save_report, save_figure, tokenize_function

import pandas as pd
import numpy as np
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
from torch import nn
from datasets import Dataset, DatasetDict
from transformers import DataCollatorWithPadding

In [ ]:
import re

def clean_clinical_text(text):
    """
    Cleans raw MIMIC-III discharge summaries for NLP processing.
    """
    if not isinstance(text, str):
        return ""
    
    text = text.replace('___', '')
    
    text = re.sub(r'Name:.*?Unit No:.*?\n', '', text)
    text = re.sub(r'Admission Date:.*?Discharge Date:.*?\n', '', text)
    text = re.sub(r'Date of Birth:.*?Sex:.*?\n', '', text)
    
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

### Library Version Check

Print the installed versions of `accelerate` and `transformers` to confirm reproducibility of the training environment.

In [30]:
import accelerate
import transformers
print(f"Accelerate version: {accelerate.__version__}")
print(f"Transformers version: {transformers.__version__}")

Accelerate version: 1.12.0
Transformers version: 5.1.0


## 3. Hyperparameter Configuration

All key training hyperparameters are defined here as constants for easy reproducibility and experimentation:

| Parameter | Value | Description |
|---|---|---|
| `MODEL_NAME` | `nlpie/tiny-biobert` | Pre-trained checkpoint from Hugging Face Hub |
| `MAX_LEN` | 512 | Maximum token sequence length (BERT upper bound) |
| `BATCH_SIZE` | 32 | Training batch size per device |
| `EPOCHS` | 3 | Number of fine-tuning epochs |
| `LEARNING_RATE` | 2e-5 | AdamW learning rate (common for BERT fine-tuning) |
| `POS_WEIGHT_VALUE` | 5.0 | Initial positive-class weight (overridden dynamically below) |

In [31]:
MODEL_NAME = "nlpie/tiny-biobert"
MAX_LEN = 512
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-5
POS_WEIGHT_VALUE = 5.0

### Device Selection

Training is automatically routed to a CUDA GPU when available, otherwise falls back to CPU.

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

Training on device: cuda


## 4. Data Loading & Preprocessing

Two raw data files are used:
- **`ami_cohort_discharge_notes.csv`** — de-identified clinical discharge summaries (NLP input)
- **`ami_cohort_structured_features.csv`** — structured ICU features including the target label `hospital_expire_flag`

The two sources are joined on `hadm_id` (hospital admission ID) to produce a single dataframe containing the note text and the binary mortality label.

In [33]:
structured_path = os.path.join(raw_data_path, 'ami_cohort_structured_features.csv')
nlp_features_path = os.path.join(processed_data_path, 'cleaned_extracted_notes.csv')

In [34]:
df_notes = pd.read_csv(nlp_features_path)
df_struct = pd.read_csv(structured_path)
df_notes = df_notes.rename(columns={'extracted_text': 'text'})
df_notes = df_notes.drop(columns=['label'])

df = pd.merge(df_notes, df_struct[['hadm_id', 'hospital_expire_flag']], on='hadm_id', how='inner')

### Column Renaming & Cleaning

Standardise column names to `text` and `label`, then drop rows with missing values to ensure clean inputs to the tokeniser.

In [35]:
df = df.rename(columns={'text': 'text', 'hospital_expire_flag': 'label'})
df = df[['hadm_id', 'text', 'label']].dropna()

### Data Preview

Inspect the first few rows to verify the merge, column structure, and label values before training.

In [36]:
df.head()

,hadm_id,text,label
0,27897940,Mr. is an with history of AAA s/p repair compl...,0
1,26913865,"female with , HTN, diabetes, CKD presented wit...",0
2,24947999,"year old female with history of HTN, CVA, CAD ...",0
3,25242409,Outpatient Providers: with PMHx significant fo...,0
4,25911675,Ms. is a year old woman with a past medical hi...,0


### Type Coercion & Class Distribution

The `text` column is explicitly cast to `str` to guard against any numeric or `NaN` entries. The class distribution is then printed to quantify the degree of imbalance — **mortality events (label=1) are rare**, motivating the weighted loss approach below.

In [37]:
df['text'] = df['text'].astype(str)

In [38]:
print(f"Data Loaded. Shape: {df.shape}")
print(f"Class Distribution:\n{df['label'].value_counts()}")

Data Loaded. Shape: (27677, 3)
Class Distribution:
label
0    26340
1     1337
Name: count, dtype: int64


## 5. Tokenisation

The merged dataframe is converted to a Hugging Face `Dataset`, the label column is encoded as a proper class label (required for stratified splitting), and the data is split **80/20** into training and held-out test sets with `stratify_by_column="label"` to preserve the class ratio in both splits.

The tiny-biobert tokeniser is then loaded and applied in a batched map operation. Post-tokenisation cleanup:
- The raw `text` column is removed (weights are in token IDs)
- The `label` column is renamed to `labels` (Hugging Face `Trainer` convention)
- The dataset is cast to PyTorch tensors

In [39]:
df['text'] = df['text'].apply(lambda x: x[-2500:] if isinstance(x, str) else x)

In [40]:
dataset = Dataset.from_pandas(df[['text', 'label']])

In [41]:
dataset = dataset.class_encode_column("label")

Casting to class labels: 100%|██████████| 27677/27677 [00:00<00:00, 182933.20 examples/s]


In [42]:
dataset = dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")

In [43]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [44]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LEN)

In [45]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 5536/5536 [00:03<00:00, 1406.58 examples/s]


In [46]:
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

## 6. Class-Imbalance Handling — Positive-Class Weighting

The dataset is **highly imbalanced** (~26 340 surviving vs ~1 337 deceased). To prevent the model from simply predicting the majority class, a **`BCEWithLogitsLoss`** weighted by the negative-to-positive ratio is used:

$$\text{pos\_weight} = \frac{N_{\text{negative}}}{N_{\text{positive}}}$$

This upweights gradient contributions from positive (mortality) examples during backpropagation, improving sensitivity for the rare class.

In [47]:
train_labels = tokenized_datasets['train']['labels']
num_pos = sum(train_labels)
num_neg = len(train_labels) - num_pos

In [48]:
if num_pos > 0:
    ratio = float(num_neg / num_pos)
    pos_weight_value = ratio 
else:
    pos_weight_value = 1.0

In [49]:
pos_weight_tensor = torch.tensor([pos_weight_value], device=device)

### Custom `WeightedTrainer`

`WeightedTrainer` subclasses Hugging Face `Trainer` and overrides `compute_loss` to apply `BCEWithLogitsLoss` with `pos_weight`. The logit for the positive class (index 1 of the 2-class output) is extracted and compared against the float-cast labels.

In [50]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Use the global pos_weight_tensor
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
        loss = loss_fct(logits.view(-1, 2)[:, 1], labels.float())
        
        return (loss, outputs) if return_outputs else loss

### Evaluation Metrics

The `compute_metrics` callback is passed to the `Trainer` and evaluated at the end of each epoch. Given the class imbalance, four complementary metrics are reported:

| Metric | Rationale |
|---|---|
| **Accuracy** | Overall correctness; misleadingly high in imbalanced settings |
| **F1 (binary)** | Harmonic mean of precision and recall — primary metric |
| **Recall** | Sensitivity for the positive class (critical in clinical risk prediction) |
| **AUROC** | Rank-order discrimination; threshold-independent |

Predicted probabilities are derived via softmax over the raw logits.

In [51]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    probs = torch.nn.functional.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()
    
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = 0.0
        
    return {'accuracy': acc, 'f1': f1, 'recall': recall, 'auc': auc}

## 7. Model Initialisation

The `tiny-biobert` checkpoint is loaded with a randomly initialised 2-class classification head (`num_labels=2`) and moved to the selected device. The `UNEXPECTED` weights in the load report correspond to the pre-training MLM head, which is correctly discarded for sequence classification.

In [52]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

Loading weights: 100%|██████████| 69/69 [00:00<00:00, 380.70it/s, Materializing param=bert.encoder.layer.3.output.dense.weight]              
BertForSequenceClassification LOAD REPORT from: nlpie/tiny-biobert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight

## 8. Training Configuration & Fine-Tuning

A dynamic padding collator is used so that sequences within each batch are padded to the length of the longest sequence, rather than globally to `MAX_LEN`, saving memory and compute.

`TrainingArguments` key settings:
- **`fp16=True`** — mixed-precision training (automatically enabled only when CUDA is available) for ~2× speedup
- **`warmup_steps=100`** — gradual LR warm-up to stabilise early training
- **`weight_decay=0.01`** — AdamW L2 regularisation
- **`load_best_model_at_end=True`** — restores the checkpoint with the best validation loss after training completes

In [53]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [54]:
training_args = TrainingArguments(
    output_dir='../results/bert_finetuned',
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE*2,
    warmup_steps=100,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),  # Auto-enable Mixed Precision for speed
    logging_steps=50,
    eval_strategy="epoch",  # Updated name
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

In [55]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

### Training Run

Launch fine-tuning. The Trainer logs metrics at every epoch end, showing the evolution of validation loss, F1, recall, and AUROC. Training took ~46 minutes across 3 epochs on this GPU.

| Epoch | Train Loss | Val Loss | Accuracy | F1 | Recall | AUROC |
|---|---|---|---|---|---|---|
| 1 | 1.290 | 1.083 | 0.826 | 0.246 | 0.588 | 0.778 |
| 2 | 0.971 | 1.059 | 0.873 | 0.304 | 0.573 | 0.805 |
| 3 | 1.054 | 1.024 | 0.863 | 0.305 | 0.625 | 0.810 |

In [56]:
print("\nStarting Fast Fine-Tuning...")
trainer.train()


Starting Fast Fine-Tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Recall,Auc
1,0.535253,0.589418,0.988439,0.870445,0.805243,0.988038
2,0.427339,0.340358,0.994581,0.942085,0.913858,0.991313
3,0.317285,0.335750,0.994762,0.944338,0.921348,0.992432


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.58it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=2076, training_loss=0.3784481126219321, metrics={'train_runtime': 2380.7505, 'train_samples_per_second': 27.9, 'train_steps_per_second': 0.872, 'total_flos': 952439146186752.0, 'train_loss': 0.3784481126219321, 'epoch': 3.0})

## 9. Model Persistence

The best fine-tuned model weights and the tokeniser vocabulary are saved to disk at `results/model/finetuned_tiny_clinicalbert/`. This directory is referenced by downstream notebooks that use the model for inference or ensemble feature extraction.

In [57]:
model_path = os.path.join(results_path, "model/finetuned_tiny_clinicalbert")
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model saved to {model_path}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.90it/s]

Model saved to E:\IIT\IRP_NEW\results\model/finetuned_tiny_clinicalbert


## 10. NLP Risk Score Generation

After fine-tuning, the model is run in inference mode over the **full cohort** (all 27 677 admissions) to produce a continuous **NLP-derived mortality risk score** (`nlp_finetuned_risk_score`) — the softmax probability for the positive class.

This probability vector forms the NLP feature column that is passed to the downstream ensemble model (e.g., XGBoost / LightGBM) in later notebooks.

In [58]:
print("\nGenerating Risk Scores for full dataset...")
full_dataset = Dataset.from_pandas(df[['hadm_id', 'text', 'label']])
full_dataset = full_dataset.class_encode_column("label")


Generating Risk Scores for full dataset...


Casting to class labels: 100%|██████████| 27677/27677 [00:00<00:00, 204339.63 examples/s]


In [59]:
full_tokenized = full_dataset.map(tokenize_function, batched=True)
full_tokenized = full_tokenized.remove_columns(["text", "hadm_id", "label"])

Map: 100%|██████████| 27677/27677 [00:18<00:00, 1482.18 examples/s]


### Batch Inference

The `Trainer.predict()` method runs the full dataset through the model. Raw logits are converted to probabilities via softmax, and the positive-class probability (index 1) is extracted as the risk score.

In [60]:
predictions = trainer.predict(full_tokenized)
probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1)

In [61]:
df['nlp_finetuned_risk_score'] = probs[:, 1].numpy()

## 11. Output Persistence

The generated risk scores are saved as `nlp_finetuned_features.csv` in the processed data directory. Only the `hadm_id` and `nlp_finetuned_risk_score` columns are exported, keeping the file lightweight for downstream merging.

In [62]:
output_path =  os.path.join(processed_data_path, "nlp_finetuned_features.csv")
df[['hadm_id', 'nlp_finetuned_risk_score']].to_csv(output_path, index=False)

In [63]:
print(f"New features saved to: {output_path}")
print(df[['hadm_id', 'label', 'nlp_finetuned_risk_score']].head())

New features saved to: E:\IIT\IRP_NEW\data\processed\nlp_finetuned_features.csv
    hadm_id  label  nlp_finetuned_risk_score
0  27897940      0                  0.002957
1  26913865      0                  0.002794
2  24947999      0                  0.002973
3  25242409      0                  0.002775
4  25911675      0                  0.002812
